In [ ]:
import time
from datetime import datetime
from io import BytesIO
from pathlib import Path
import requests
from PIL import Image
import math

from tile_utils import latlon_to_tile, latlon_to_pixel, draw_marker, draw_area_box


# Provider configuration: URL template, typical x/y ordering, typical max zoom,
# and recommended pause between requests (each service has its own usage policy).
PROVIDERS = {
    "opentopomap": {
        "url_template": "https://tile.opentopomap.org/{z}/{x}/{y}.png",
        "max_zoom": 17,
        "sleep": 0.5,  # Seconds to wait between tile requests (rate limiting)
    },
    "esri": {
        # NOTE: ESRI uses {z}/{y}/{x} ordering, reversed from the standard {z}/{x}/{y}
        # This is a common gotcha when switching between tile providers
        "url_template": "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        "max_zoom": 17,
        "sleep": 0.2,  # Seconds to wait between tile requests (rate limiting)
    },
}


def get_point_map(lat, lon, area_meters=56, provider="esri", zoom=None, radius=2, 
                  out_prefix=None, output_dir="../img/maps", show_marker=True):
    """
    Downloads map tiles around a point and composites them into a single image,
    optionally marking the exact location and area of interest.
    
    Args:
        lat: Latitude of the center point in degrees
        lon: Longitude of the center point in degrees
        area_meters: Buffer radius in meters for drawing the area box
                     (same value used in reduceRegion calculations)
        provider: "esri" (real satellite imagery) or "opentopomap" (topographic/elevation map)
        zoom: Zoom level; if not specified, uses the provider's typical max_zoom
        radius: Number of tiles to add around the center in each direction
                (e.g., radius=2 gives a 5x5 tile grid)
        out_prefix: Base name for the output file; if not specified, uses the provider name
        output_dir: Directory where the image is saved (created automatically if it doesn't exist)
        show_marker: If True, draws a red dot at the exact location and an area box
    
    Returns:
        pathlib.Path: Path to the saved image file
    
    Raises:
        ValueError: If the provider is not recognized in the PROVIDERS dictionary
    """
    # Validate that the requested provider exists in our configuration
    if provider not in PROVIDERS:
        raise ValueError(f"Provider '{provider}' not recognized. Options: {list(PROVIDERS.keys())}")

    # Load provider-specific configuration
    config = PROVIDERS[provider]
    zoom = zoom or config["max_zoom"]  # Use max zoom if not explicitly provided
    out_prefix = out_prefix or f"point_{provider}"  # Default filename prefix

    # Generate a unique filename with timestamp to avoid overwriting previous maps
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.png"
    
    # Create output directory if it doesn't exist
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    # Calculate the tile coordinates for the center point
    # These are integer tile indices at the given zoom level
    x_center, y_center = latlon_to_tile(lat, lon, zoom)
    
    # Define the bounding box of tiles to download
    x_min, x_max = x_center - radius, x_center + radius
    y_min, y_max = y_center - radius, y_center + radius

    # Standard tile size for Web Mercator tiles (256x256 pixels)
    tile_size = 256
    
    # Calculate the total canvas dimensions in pixels
    # +1 because range is inclusive of both min and max
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    
    # Create a blank RGB canvas to composite all tiles onto
    canvas = Image.new("RGB", (width, height))

    # Set a proper User-Agent header as good practice for API usage
    # Replace with your actual email for responsible API consumption
    headers = {"User-Agent": "agri_land_suitability_pipeline (your_real_email@domain.com)"}

    # Download and stitch together all tiles in the bounding box
    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            # Build the tile URL using the provider's template
            url = config["url_template"].format(z=zoom, x=x, y=y)
            
            # Fetch the tile image with a timeout to avoid hanging
            resp = requests.get(url, headers=headers, timeout=10)

            # Skip this tile if the server returns an error
            if resp.status_code != 200:
                print(f"Tile {x},{y} failed with status {resp.status_code}")
                time.sleep(config["sleep"])
                continue

            try:
                # Open the downloaded image from memory (avoids writing temp files)
                tile_img = Image.open(BytesIO(resp.content))
                
                # Paste the tile onto the canvas at the correct pixel offset
                # (x - x_min) and (y - y_min) convert tile coordinates to canvas pixel coordinates
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"Tile {x},{y} failed: {e}")

            # Respect the provider's rate limiting to avoid being blocked
            time.sleep(config["sleep"])

    # Draw the point marker and area box on the composited canvas
    if show_marker:
        # Get the absolute world pixel coordinates for the given lat/lon
        px_world, py_world = latlon_to_pixel(lat, lon, zoom, tile_size)
        
        # Convert world pixel coordinates to canvas-local pixel coordinates
        # by subtracting the offset of the top-left tile in our grid
        px_canvas = px_world - x_min * tile_size
        py_canvas = py_world - y_min * tile_size
        
        # Draw a red circular marker at the exact point location
        draw_marker(canvas, px_canvas, py_canvas)
        
        # Draw a box representing the area used in reduceRegion calculations
        # This shows the actual spatial extent of the analysis
        draw_area_box(canvas, px_canvas, py_canvas, area_meters, lat=lat, zoom=zoom)

    # Save the final composited image to disk
    canvas.save(out_path)
    print(f"Image saved to {out_path} ({width}x{height}px)")
    return out_path


if __name__ == "__main__":
    # Example usage: generate a map with real satellite imagery
    
    # Convert hectares to the equivalent buffer radius in meters
    # area_meters = sqrt(hectares * 10000) / 2
    # For 2 hectares: sqrt(20000) / 2 ≈ 70.71 / 2 ≈ 35.36 meters radius
    ha = 2
    area_meters = (math.sqrt(ha * 10000)) / 2
    
    # Test point coordinates (Colombia)
    Latitude, Longitude = 7.4584221918243045,-73.222052853104
    
    # Generate satellite imagery map (ESRI World Imagery)
    get_point_map(Latitude, Longitude, area_meters, provider="esri", radius=2)

    # Generate topographic/elevation map (OpenTopoMap)
    # Uncomment if you need it again later
    get_point_map(Latitude, Longitude, area_meters, provider="opentopomap", radius=2)

# Reference points for quick access (commented out location names with coordinates):
# El Playon         --||     7.4584221918243045,    -73.222052853104
# Finca Matanza     --||     7.300921,              -73.009794
# Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
# Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223

Imagen guardada en ../img/maps/point_esri-v260805174215.png (1280x1280px)
Imagen guardada en ../img/maps/point_opentopomap-v260805174222.png (1280x1280px)
